#### Learning Rules - FinDok Dataset - First Experiments

In [29]:
import argparse
import json
import os
import re
from datetime import datetime
from pathlib import Path

import rulechef
from openai import OpenAI
from rulechef import RuleChef, Task, TaskType
from rulechef.coordinator import AgenticCoordinator
from rulechef.core import Dataset, Example, RuleFormat
from rulechef.evaluation import evaluate_rules_individually, print_rule_metrics
from rulechef.executor import RuleExecutor

from clear_anonymization.ner_datasets import get_dataset_class_definitions
from clear_anonymization.ner_datasets.ner_dataset import NERData, NERSample

In [5]:
import tempfile

from openai import OpenAI
from rulechef import RuleChef, RuleFormat, Task, TaskType

from clear_anonymization.utils.utils import NEROutput, get_class_def, sample_data

In [119]:
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table
from rich.text import Text

console = Console()

# Entity type colors for NER highlighting
_ENTITY_COLORS = {
    "DRUG": "bold cyan",
    "DOSAGE": "bold yellow",
    "FREQUENCY": "bold green",
    "CONDITION": "bold magenta",
}
_FALLBACK_COLORS = [
    "bold cyan",
    "bold yellow",
    "bold green",
    "bold magenta",
    "bold red",
    "bold blue",
]


def _color_for(entity_type, seen=None):
    """Get a consistent color for an entity type."""
    if seen is None:
        seen = {}
    if entity_type in _ENTITY_COLORS:
        return _ENTITY_COLORS[entity_type]
    if entity_type not in seen:
        seen[entity_type] = _FALLBACK_COLORS[len(seen) % len(_FALLBACK_COLORS)]
    return seen[entity_type]


def _f1_color(val):
    if val >= 0.8:
        return "green"
    if val >= 0.5:
        return "yellow"
    return "red"


def show_rules(chef):
    """Show all rules in a compact table."""
    table = Table(title="Rules", show_lines=False)
    table.add_column("#", style="dim", width=3)
    table.add_column("Name", max_width=35)
    table.add_column("Format", width=6)
    table.add_column("Pattern", max_width=55)
    table.add_column("Type", width=12)
    table.add_column("Pri", width=3, justify="right")
    table.add_column("Conf", width=5, justify="right")

    for i, rule in enumerate(chef.dataset.rules):
        entity_type = ""
        if rule.output_template:
            entity_type = rule.output_template.get(
                "type", rule.output_template.get("label", "")
            )
        pattern = rule.content[:55] + "..." if len(rule.content) > 55 else rule.content
        conf = rule.confidence
        conf_style = "green" if conf >= 0.7 else "yellow" if conf >= 0.4 else "red"

        table.add_row(
            str(i),
            rule.name,
            rule.format.value,
            pattern,
            entity_type,
            str(rule.priority),
            f"[{conf_style}]{conf:.2f}[/]",
        )
    console.print(table)


def show_rule(chef, index):
    """Deep dive into a single rule by index."""
    rules = chef.dataset.rules
    if isinstance(index, str):
        rule = next((r for r in rules if r.id == index), None)
        if not rule:
            console.print(f"[red]Rule '{index}' not found[/]")
            return
    else:
        if index >= len(rules):
            console.print(
                f"[red]Index {index} out of range (have {len(rules)} rules)[/]"
            )
            return
        rule = rules[index]

    # Header info
    entity_type = ""
    if rule.output_template:
        entity_type = rule.output_template.get(
            "type", rule.output_template.get("label", "")
        )
    header = f"[bold]{rule.name}[/]  |  {rule.format.value}  |  priority={rule.priority}  |  conf={rule.confidence:.2f}"
    if entity_type:
        header += f"  |  type={entity_type}"

    # Pattern with syntax highlighting
    if rule.format.value == "regex":
        syntax = Syntax(rule.content, "perl", theme="monokai", word_wrap=True)
    elif rule.format.value == "code":
        syntax = Syntax(rule.content, "python", theme="monokai", word_wrap=True)
    else:
        syntax = Syntax(rule.content, "json", theme="monokai", word_wrap=True)

    # Stats
    stats = f"Applied: {rule.times_applied}  |  Successes: {rule.successes}  |  Failures: {rule.failures}"
    if rule.times_applied > 0:
        rate = rule.successes / rule.times_applied * 100
        stats += f"  |  Success rate: {rate:.0f}%"

    # Build panel content
    content = Text()
    content.append(stats + "\n\n")

    panel = Panel.fit(
        syntax,
        title=header,
        subtitle=stats,
        border_style="blue",
    )
    console.print(panel)

    # Output template
    if rule.output_template:
        console.print(f"  Output template: {rule.output_template}")


def show_eval(eval_result):
    """Show evaluation results with color-coded per-class metrics."""
    if not eval_result or eval_result.total_docs == 0:
        console.print("[dim]No evaluation results.[/]")
        return

    # Summary
    f1_style = _f1_color(eval_result.micro_f1)
    summary = Text()
    summary.append("Micro F1: ")
    summary.append(f"{eval_result.micro_f1:.1%}", style=f"bold {f1_style}")
    summary.append(f"  |  Macro F1: {eval_result.macro_f1:.1%}")
    summary.append(f"  |  Exact match: {eval_result.exact_match:.1%}")
    summary.append(
        f"  |  P={eval_result.micro_precision:.1%}  R={eval_result.micro_recall:.1%}"
    )
    summary.append(f"  |  {eval_result.total_docs} docs")
    console.print(summary)

    # Per-class table
    if eval_result.per_class:
        table = Table(show_lines=False)
        table.add_column("Class", min_width=15)
        table.add_column("F1", justify="right", width=6)
        table.add_column("Prec", justify="right", width=6)
        table.add_column("Recall", justify="right", width=6)
        table.add_column("TP", justify="right", width=4, style="green")
        table.add_column("FP", justify="right", width=4, style="red")
        table.add_column("FN", justify="right", width=4, style="yellow")

        for cm in sorted(eval_result.per_class, key=lambda c: c.f1, reverse=True):
            f1_style = _f1_color(cm.f1)
            table.add_row(
                cm.label,
                f"[{f1_style}]{cm.f1:.0%}[/]",
                f"{cm.precision:.0%}",
                f"{cm.recall:.0%}",
                str(cm.tp),
                str(cm.fp),
                str(cm.fn),
            )
        console.print(table)


def show_failures(eval_result, entity_type=None):
    """Show what the FPs and FNs actually are.

    Args:
        eval_result: EvalResult from chef.evaluate() or evaluate_dataset()
        entity_type: Optional filter — only show failures involving this type (e.g. "FREQUENCY")
    """
    if not eval_result.failures:
        console.print("[green]No failures — all examples matched![/]")
        return

    shown = 0
    for failure in eval_result.failures:
        inp = failure["input"]
        expected = failure["expected"]
        got = failure["got"]
        text = inp.get("text", str(inp))

        # Get entity lists (NER/extraction) or labels (classification)
        expected_entities = expected.get("entities", expected.get("spans", []))
        got_entities = got.get("entities", got.get("spans", []))

        # Classification: simple expected vs got label
        if not expected_entities and not got_entities:
            exp_label = expected.get("label", "")
            got_label = got.get("label", "")
            if entity_type and entity_type not in (exp_label, got_label):
                continue
            shown += 1
            t = Text()
            t.append(f"{shown}. ", style="dim")
            t.append(text)
            t.append("\n   expected: ", style="dim")
            t.append(exp_label, style="green")
            t.append("   got: ", style="dim")
            t.append(got_label or "(no match)", style="red")
            console.print(t)
            continue

        # NER: diff expected vs got to find FPs and FNs
        # Use list-based matching (not sets) so duplicates are caught correctly
        def _key(e):
            return (e.get("text", "").lower(), e.get("type", e.get("label", "")))

        remaining_expected = [_key(e) for e in expected_entities]
        fps = []
        for e in got_entities:
            k = _key(e)
            if k in remaining_expected:
                remaining_expected.remove(k)
            else:
                fps.append(e)

        remaining_got = [_key(e) for e in got_entities]
        fns = []
        for e in expected_entities:
            k = _key(e)
            if k in remaining_got:
                remaining_got.remove(k)
            else:
                fns.append(e)

        # Apply entity_type filter
        if entity_type:
            fps = [e for e in fps if e.get("type", e.get("label", "")) == entity_type]
            fns = [e for e in fns if e.get("type", e.get("label", "")) == entity_type]
            if not fps and not fns:
                continue

        shown += 1
        t = Text()
        t.append(f"{shown}. ", style="dim")
        t.append(text, style="white")
        console.print(t)

        # Track matched keys to detect duplicates
        matched_keys = [
            _key(e) for e in got_entities if _key(e) not in [_key(f) for f in fps]
        ]

        for e in fps:
            etype = e.get("type", e.get("label", "?"))
            k = _key(e)
            is_dup = k in matched_keys  # same text+type already matched as TP
            tag = "FP (duplicate)" if is_dup else "FP"
            console.print(f"   [red]{tag}[/]  \\[{etype}: {e.get('text', '?')!r}]")
        for e in fns:
            etype = e.get("type", e.get("label", "?"))
            console.print(f"   [yellow]FN[/]  \\[{etype}: {e.get('text', '?')!r}]")

    if shown == 0:
        if entity_type:
            console.print(f"[dim]No failures involving {entity_type}.[/]")
        else:
            console.print("[dim]No failures to show.[/]")
    else:
        console.print(f"\n[dim]{shown} documents with errors[/]")


def test(chef, text):
    """Extract from text and display results with highlighted entities."""
    result = chef.extract({"text": text})

    entities = result.get("entities", result.get("spans", []))
    label = result.get("label")

    if entities:
        # Build annotated text with colored entity spans
        annotated = Text()
        prev_end = 0
        for e in sorted(entities, key=lambda x: x.get("start", 0)):
            start = e.get("start", 0)
            end = e.get("end", 0)
            etype = e.get("type", "?")
            color = _color_for(etype)

            annotated.append(text[prev_end:start])
            annotated.append(f"[{etype}: ", style="dim")
            annotated.append(e.get("text", text[start:end]), style=color)
            annotated.append("]", style="dim")
            prev_end = end
        annotated.append(text[prev_end:])
        console.print(annotated)
    elif label:
        console.print(Text.assemble(text, "  →  ", (label, "bold cyan")))
    else:
        console.print(Text.assemble(text, "  →  ", ("(no match)", "dim")))


print("Display helpers loaded: show_rules, show_rule, show_eval, show_failures, test")


def show_rules_eval(chef, max_examples: int = 3, verbose=True):
    executor = RuleExecutor()

    metrics = evaluate_rules_individually(
        rules=chef.dataset.rules,
        dataset=chef.dataset,
        apply_rules_fn=executor.apply_rules,
    )

    rules_to_show = sorted(metrics, key=lambda r: r.f1, reverse=True)

    for i, rule in enumerate(rules_to_show, 1):
        print(f"\n{'=' * 60}")
        print(f"{i}. {rule.rule_name} ({rule.rule_id})")
        if rule.f1 >= 0.8:
            score = "🟢 Strong"
        elif rule.f1 >= 0.5:
            score = "🟡 Medium"
        else:
            score = "🔴 Weak"

        print(f"{score}")
        print(
            f"F1: {rule.f1:.3f} | Precision: {rule.precision:.3f} | Recall: {rule.recall:.3f}"
        )

        if rule.rule_description:
            print(f"> {rule.rule_description}")

        print(rule.rule_content)
        if not rule.sample_matches:
            continue

        hits = [s for s in rule.sample_matches if s.tp > 0]
        misses = [s for s in rule.sample_matches if s.fn > 0]
        false_pos = [s for s in rule.sample_matches if s.fp > 0]

        if hits:
            print(f"\n✅ Worked ({min(len(hits), max_examples)})")
            for sample in hits[:max_examples]:
                if verbose:
                    print(f"TEXT: {sample.input['text']}")
                for pred, gold in sample.matched_pairs:
                    print(f"  ✔ {pred['text']}  →  {gold['text']}")
        if misses:
            print(f"\n❌ Missed ({min(len(misses), max_examples)})")
            for sample in misses[:max_examples]:
                if verbose:
                    print(f"TEXT: {sample.input['text']}")
                for miss in sample.missed:
                    print(f"  ✖ {miss['text']}")

        if false_pos:
            print(f"\n⚠️ False Positives ({min(len(false_pos), max_examples)})")
            for sample in false_pos[:max_examples]:
                if verbose:
                    print(f"TEXT: {sample.input['text']}")
                for fp in sample.false_positives:
                    print(f"  ⚠ {fp['text']}")

Display helpers loaded: show_rules, show_rule, show_eval, show_failures, test


In [31]:
allowed_classes = "organisation"
input_dir = Path("/share/nverdha/data/bfg/bfg_train.json")
data = NERData.from_json(json.loads(input_dir.read_text()))
samples = sample_data(data.samples, allowed_classes)


def get_openai_client() -> OpenAI:
    """Get OpenAI client configured from environment variables."""
    api_key = os.getenv("OPENAI_API_KEY") or "EMPTY"
    base_url = os.getenv("OPENAI_BASE_URL") or "http://localhost:8000/v1"

    return OpenAI(api_key=api_key, base_url=base_url)


client = get_openai_client()

date_str = datetime.now().strftime("%Y-%m-%d")
classes_str, allowed_classes_def = get_class_def("bfg", [allowed_classes])
task = Task(
    name="Named Entity Recognition",
    description=f"Extract {allowed_classes_def} from text ",
    input_schema={"text": "str"},
    output_schema=NEROutput,
    type=TaskType.NER,
    text_field="text",
)
chef = RuleChef(
    task,
    client=get_openai_client(),
    dataset_name=f"{date_str}_findok_{classes_str}",
    storage_path="../rulechef_data/",
    model="openai/gpt-oss-120b",
    allowed_formats=[RuleFormat.REGEX],  # regex-only for transparency
    use_grex=True,  # use grex to suggest regex patterns from examples
)

✓ Loaded dataset: 0 corrections, 30 examples


In [8]:
for sample in samples[:30]:
    chef.add_example({"text": sample["text"]}, {"entities": sample["entities"]})

✓ Added example (buffer: 1 new, 1 total)
✓ Added example (buffer: 2 new, 2 total)
✓ Added example (buffer: 3 new, 3 total)
✓ Added example (buffer: 4 new, 4 total)
✓ Added example (buffer: 5 new, 5 total)
✓ Added example (buffer: 6 new, 6 total)
✓ Added example (buffer: 7 new, 7 total)
✓ Added example (buffer: 8 new, 8 total)
✓ Added example (buffer: 9 new, 9 total)
✓ Added example (buffer: 10 new, 10 total)
✓ Added example (buffer: 11 new, 11 total)
✓ Added example (buffer: 12 new, 12 total)
✓ Added example (buffer: 13 new, 13 total)
✓ Added example (buffer: 14 new, 14 total)
✓ Added example (buffer: 15 new, 15 total)
✓ Added example (buffer: 16 new, 16 total)
✓ Added example (buffer: 17 new, 17 total)
✓ Added example (buffer: 18 new, 18 total)
✓ Added example (buffer: 19 new, 19 total)
✓ Added example (buffer: 20 new, 20 total)
✓ Added example (buffer: 21 new, 21 total)
✓ Added example (buffer: 22 new, 22 total)
✓ Added example (buffer: 23 new, 23 total)
✓ Added example (buffer: 24 n

In [14]:
rules, eval_result = chef.learn_rules()


Learning rules from 30 training items
  Corrections: 0 (high value)
  Examples: 30
  Mode: Synthesis + Refinement (max 3 iterations)

📚 Synthesizing rules from dataset...
✓ Synthesized 5 rules (29.7s)
✓ Generated 5 rules

🔄 Refinement loop (max 3 iterations)
[1/3] Evaluating rules...
[1/3] Exact match: 0.0% (0/30), micro F1: 6.5%
[1/3] Patching 30 failures...
🩹 Synthesizing patch rules...
      Validation error: look-behind requires fixed-width pattern
   ⚠ Skipped invalid rule: Capitalised phrase after der/des
✓ Patch synthesis returned 4 rules (54.9s)
[1/3] Patched → 5 rules, exact match 0.0% → 0.0% (54.9s)
[2/3] Evaluating rules...
[2/3] Exact match: 0.0% (0/30), micro F1: 9.0%
[2/3] Patching 30 failures...
🩹 Synthesizing patch rules...
✓ Patch synthesis returned 7 rules (61.7s)
[2/3] Patched → 7 rules, exact match 0.0% → 60.0% (61.7s)
[3/3] Evaluating rules...
[3/3] Exact match: 60.0% (18/30), micro F1: 70.8%
[3/3] Patching 12 failures...
🩹 Synthesizing patch rules...
      Valida

In [21]:
show_rules_eval(chef)

Finanzamt full name
F1-Score 0.3111111111111111, Recall 0.19444444444444445, Precision 0.7777777777777778
Matches "Finanzamt" followed by its jurisdiction (e.g., Finanzamt Amstetten Melk Scheibbs).
(?<!\w)Finanzamt\s+(?:[A-Z\u00c4\u00d6\u00dc][\w&\-]*\s?)+(?!\w)(?=\b|,|\.|\n)
#########################


Company with AG or GMBH
F1-Score 0.3111111111111111, Recall 0.19444444444444445, Precision 0.7777777777777778
Matches company names ending with AG or GMBH (e.g., Vianden Robotik GMBH).
(?<!\w)(?:[A-Z\u00c4\u00d6\u00dc][\w&\-]*)(?:\s[A-Z\u00c4\u00d6\u00dc][\w&\-]*)*\s(?:AG|GMBH)(?!\w)
#########################


FA abbreviation
F1-Score 0.27272727272727276, Recall 0.16666666666666666, Precision 0.75
Matches "FA" followed by a capitalised authority name (e.g., FA Vorarlberg).
(?<!\w)FA\s+(?:[A-Z\u00c4\u00d6\u00dc][\w&\-\d]*\s?)+(?!\w)(?=\b|,|\.|\n)
#########################


Organisation after der/des ending with AG/GMBH
F1-Score 0.24390243902439027, Recall 0.1388888888888889, Precision 

In [26]:
%timeit chef.extract({"text": "Nadia arbeitet beim Finanzamt Xy GMBH"})

24.7 μs ± 3.3 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [27]:
chef.extract({"text": "Nadia arbeitet beim Xy GMBH"})

{'entities': [{'text': 'Xy GMBH',
   'start': 20,
   'end': 27,
   'type': 'organisation',
   'rule_id': '74fb100a',
   'rule_name': 'Company with AG or GMBH'}]}

In [23]:
show_rules(chef)

                                                       Rules                                                       
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┓
┃ #   ┃ Name                             ┃ Format ┃ Pattern                          ┃ Type         ┃ Pri ┃  Conf ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━┩
│ 0   │ FA abbreviation                  │ regex  │ (?<!\w)FA\s+(?:[A-Z\u00c4\u00d6… │ organisation │  10 │  0.50 │
│ 1   │ Finanzamt full name              │ regex  │ (?<!\w)Finanzamt\s+(?:[A-Z\u00c… │ organisation │   9 │  0.50 │
│ 2   │ Company with AG or GMBH          │ regex  │ (?<!\w)(?:[A-Z\u00c4\u00d6\u00d… │ organisation │   8 │  0.50 │
│ 3   │ Hyphenated organisation names    │ regex  │ (?<!\w)[A-Z\u00c4\u00d6\u00dc][… │ organisation │   7 │  0.50 │
│ 4   │ Capitalised phrase after der/des │ regex  │ (?<=\b(?:der|des)\s)(?!\b(?:Bes… │ organisation │   5 │  0.50 │
│ 5   │ FA after der/des                 │ regex  │ (?<=\b(?:der|des)\s)FA\s+(?:[A-… │ organisation │   9 │  0.50 │
│ 6   │ Organisation after der/des       │ regex  │ (?<=\b(?:der|des)\s)([A-Z\u00c4… │ organisation │   6 │  0.50 │
│     │ ending with AG/GMBH              │        │                                  │              │     │       │
└─────┴──────────────────────────────────┴────────┴──────────────────────────────────┴──────────────┴─────┴───────┘

In [24]:
show_failures(eval_result, "organisation")

1. rch die Richterin Priv.-Doz.in Roswitha Cofala  in der Beschwerdesache Waldemar Harmann, 
Hoheneggerstraße 17, 4870 Mörasing, Österreich, über die Beschwerde vom 24. April 2017 gegen den Bescheid des 
Finanzamt Amstetten Melk Scheibbs 
vom 28. März 2017 betreffend Einkommensteuer (Arbeitnehmerveranlagung) 2011 
Steuernummer zu Recht erkannt:  
Der Beschwerde wird gemäß § 279 BAO teilweise Folge gegeben. 
Der angefochtene Bescheid 

FP  [organisation: 'Finanzamt Amstetten Melk Scheibbs ']

FN  [organisation: 'Finanzamt Amstetten Melk Scheibbs']

2.   
GZ. RV/7103005/2013 
  
 
 
 
IM NAMEN DER REPUBLI K 
Das Bundesfinanzgericht hat durch den Richter M. in der Beschwerdesache Bauermeister Getränke, Zur Piesting 7, 8682
Hönigsberg, Österreich, diese vertreten durch Mag. Dieter Walla & Partner Steuerberater OG, Kremser 
Landstraße 7, 3100 Sankt Pölten, über die Beschwerde vom 2. August 2013 gege

FN  [organisation: 'Bauermeister Getränke']

...

11. beratungskanzlei, der der beschwerdegegenständliche 
Bescheid zugestellt worden ist, vom 22.7.2019 wurde wie folgt ausgeführt: 
"Der Einkommensteuerbescheid für das Jahr 2009 vom 13.7.2011 betreffend Trafenfen Automotive  wurde 
Ihnen als steuerlicher Vertreter zugestellt. 
5 von 16
Seite 6 von 16 
 
 
Sie werden nun aufgefordert, diesen Bescheid mit Eingangsstempel der Kanzlei in Kopie 
einzureichen. 
Sollte kein Ei

FN  [organisation: 'Trafenfen Automotive']

12.  
Mit Schreiben vom 25.2.2020 wurde an DI Zeuge2 eine schriftliche Zeugeneinvernahme zum 
Beweisthema AfA-Satz von 3% für die auf der Liegenschaft EZGST bestehenden Gebäude laut 
den Schreiben an die Trafenfen Automotive  GmbH vom 14.11.2011 und vom 29.2.2012 versendet. 
Ausgeführt wurde wie folgt: 
„Mit Schreiben vom 21.2.2020 wurde vom Beschwerdeführer Ihre Einvernahme als Zeuge im 
gegenständlichen Verfahren beant

FN  [organisation: 'Trafenfen Automotive']

12 documents with errors

In [25]:
show_eval(eval_result)

Micro F1: 70.8%  |  Macro F1: 70.8%  |  Exact match: 60.0%  |  P=79.3%  R=63.9%  |  30 docs

┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━┓
┃ Class           ┃     F1 ┃   Prec ┃ Recall ┃   TP ┃   FP ┃   FN ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━┩
│ organisation    │    71% │    79% │    64% │   23 │    6 │   13 │
└─────────────────┴────────┴────────┴────────┴──────┴──────┴──────┘

### Improving the rules

In [35]:
# there are some duplicate rules which require some prunning

chef.coordinator = AgenticCoordinator(client, model="openai/gpt-oss-120b")


print(f"Rules before pruning: {len(chef.dataset.rules)}")


chef.coordinator = AgenticCoordinator(
    client, model="openai/gpt-oss-120b", prune_after_learn=True
)

rules, eval_result = chef.learn_rules(max_refinement_iterations=3)

print(f"\nRules after pruning: {len(chef.dataset.rules)}")

Rules before pruning: 7

Learning rules from 30 training items
  Corrections: 0 (high value)
  Examples: 30
  Mode: Synthesis + Refinement (max 3 iterations)

📚 Synthesizing rules from dataset...
✓ Synthesized 4 rules (27.9s)
✓ Generated 4 rules

🔄 Refinement loop (max 3 iterations)
[1/3] Evaluating rules...
[1/3] Exact match: 6.7% (2/30), micro F1: 23.8%
🤖 Coordinator: Prioritize generating rules that capture organization-specific cues (e.g., corporate suffixes, department names, legal entity terms) while filtering out generic words that cause many false positives. Avoid overly broad patterns that match common nouns.
   Focus: organisation
[1/3] Patching 28 failures...
🩹 Synthesizing patch rules...
      Validation error: look-behind requires fixed-width pattern
   ⚠ Skipped invalid rule: Company_AG_GMBH
✓ Patch synthesis returned 4 rules (55.8s)
[1/3] Patch made it worse (6.7% → 0.0%), keeping previous
[2/3] Evaluating rules...
[2/3] Exact match: 6.7% (2/30), micro F1: 23.8%
🤖 Coordi

In [36]:
show_rules(chef)

                                                       Rules                                                       
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┓
┃ #   ┃ Name                             ┃ Format ┃ Pattern                          ┃ Type         ┃ Pri ┃  Conf ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━┩
│ 0   │ FA_abbreviation                  │ regex  │ (?<!\w)FA\s+(?:[A-ZÄÖÜ][\w&\-]*… │ organisation │  10 │  0.50 │
│ 1   │ Finanzamt_full                   │ regex  │ (?<!\w)Finanzamt\s+(?:[A-ZÄÖÜ][… │ organisation │   9 │  0.50 │
│ 2   │ Company_AG_GMBH                  │ regex  │ (?<!\w)(?:[A-ZÄÖÜ][\w&\-\u00C0-… │ organisation │   8 │  0.50 │
│ 3   │ Capitalised_phrase_after_der_des │ regex  │ (?<=\b(?:der|des)\s)(?!\b(?:Bes… │ organisation │   5 │  0.50 │
│ 4   │ Literal_Bauermeister_Getranke    │ regex  │ \bBauermeister\s+Getranke\b      │ organisation │  10 │  0.50 │
│ 5   │ Literal_TalPflege                │ regex  │ \bTalPflege\b                    │ organisation │  10 │  0.50 │
└─────┴──────────────────────────────────┴────────┴──────────────────────────────────┴──────────────┴─────┴───────┘

In [37]:
show_eval(eval_result)

Micro F1: 76.5%  |  Macro F1: 76.5%  |  Exact match: 66.7%  |  P=81.2%  R=72.2%  |  30 docs

┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━┓
┃ Class           ┃     F1 ┃   Prec ┃ Recall ┃   TP ┃   FP ┃   FN ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━┩
│ organisation    │    76% │    81% │    72% │   26 │    6 │   10 │
└─────────────────┴────────┴────────┴────────┴──────┴──────┴──────┘

In [38]:
show_failures(eval_result, "organisation")

1.   
GZ. RV/7103005/2013 
  
 
 
 
IM NAMEN DER REPUBLI K 
Das Bundesfinanzgericht hat durch den Richter M. in der Beschwerdesache Bauermeister Getränke, Zur Piesting 7, 8682
Hönigsberg, Österreich, diese vertreten durch Mag. Dieter Walla & Partner Steuerberater OG, Kremser 
Landstraße 7, 3100 Sankt Pölten, über die Beschwerde vom 2. August 2013 gege

FN  [organisation: 'Bauermeister Getränke']

2. lässig. 
 
Entscheidungsgründe 
Zum Erkenntnis: Mit Bescheid des Finanzamtes Lilienfeld St. Pölten vom 7. Mai 2013 wurden 
die Anspruchszinsen 2007 für die Einkommensteuernachforderung 2007 von Herrn Bauermeister Getränke, 
nunmehr Bauermeister Getränke (in weiterer Folge: Bf.) in einer Höhe von € 27.080,78 festgesetzt.  
Nach verlängerter Frist zur Einbringung einer Berufung, die sich neben den Anspruchszinsen 
2007 unter anderem auch gegen die Wiederaufnahme des Feststellungsverfahrens und den 
1 von 5
Seite 2 von 5 
 
 
Feststellungsbescheid 2007 der Bauermeister Getränke  KG sowie den Einkommensteuerbescheid 2007 des 
Bf. richtet, wurde im Schriftsatz vom 2. August 2013 – soweit es die Anspruchszinsen 2007 
anbelangt – ausgeführt, dass die Bescheide über die Wiederau

FN  [organisation: 'Bauermeister Getränke']

FN  [organisation: 'Bauermeister Getränke']

...

FN  [organisation: 'Trafenfen Automotive']

10.  
Mit Schreiben vom 25.2.2020 wurde an DI Zeuge2 eine schriftliche Zeugeneinvernahme zum 
Beweisthema AfA-Satz von 3% für die auf der Liegenschaft EZGST bestehenden Gebäude laut 
den Schreiben an die Trafenfen Automotive  GmbH vom 14.11.2011 und vom 29.2.2012 versendet. 
Ausgeführt wurde wie folgt: 
„Mit Schreiben vom 21.2.2020 wurde vom Beschwerdeführer Ihre Einvernahme als Zeuge im 
gegenständlichen Verfahren beant

FP  [organisation: 'Liegenschaft EZGST']

FN  [organisation: 'Trafenfen Automotive']

10 documents with errors

In [122]:
show_rules_eval(chef, verbose=True)


1. Capitalised_phrase_after_der_des (53d24d65)
🟡 Medium
F1: 0.538 | Precision: 0.875 | Recall: 0.389
> Captures 2‑3 capitalised words that appear after "der" or "des" and are not generic nouns.
(?<=\b(?:der|des)\s)(?!\b(?:Beschwerdesache|Stadt|Gemeinde|Bezirk|Finanzamtes|Finanzamt|Bescheid|Bf|Magistrates|Beschwerdevorentscheidu)\b)((?:[A-ZÄÖÜ][\w&\-\u00C0-\u00FF]*\s){1,2}[A-ZÄÖÜ][\w&\-\u00C0-\u00FF]*)(?=\b|,)

✅ Worked (3)
TEXT: rtragsparteien eine der Bestandvertragsgebühr unterliegende Option oder ein
gebührenrechtlich nicht steuerbarer Vorvertrag vereinbart wurde:
Die Beschwerdeführerin schloss als Bestandnehmerin mit der Vianden Robotik GMBH  als
Bestandgeberin den mit 14.9.2019 datieren schriftlichen Pachtvertrag über eine
Geschäftsräumlichkeit (Betrieb einer Apotheke) ab. Der Vertrag wurde am 4.10.2019
gegenüber der belangten Behörde z
  ✔ Vianden Robotik GMBH  →  Vianden Robotik GMBH
TEXT: GZ. RV/7400382/2018
IM NAMEN DER REPUBLIK
Das Bundesfinanzgericht erkennt durch die Richt